In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

In [8]:
import os

BASE_DIR = r"C:\Users\satya\Documents\customer-segmentation"
data_path = os.path.join(BASE_DIR, 'data', 'online_retail.xlsx')

df = pd.read_excel(data_path)
print("\nFirst 5 rows:")
df.head()


First 5 rows:


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [9]:
print("Shape:", df.shape)
print("\nColumn Names:", df.columns.tolist())
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())

Shape: (541909, 8)

Column Names: ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country']

Data Types:
 InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

Missing Values:
 InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64


In [10]:
# Drop rows with missing CustomerID and Description
df_clean = df.dropna(subset=['CustomerID', 'Description'])

# Remove cancelled orders (InvoiceNo starting with 'C')
df_clean = df_clean[~df_clean['InvoiceNo'].astype(str).str.startswith('C')]

# Remove negative or zero quantity and price
df_clean = df_clean[df_clean['Quantity'] > 0]
df_clean = df_clean[df_clean['UnitPrice'] > 0]

# Convert CustomerID to integer
df_clean['CustomerID'] = df_clean['CustomerID'].astype(int)

print("Cleaned Shape:", df_clean.shape)
print("Unique Customers:", df_clean['CustomerID'].nunique())

Cleaned Shape: (397884, 8)
Unique Customers: 4338


In [11]:
# Reference date — one day after last invoice date
reference_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# Calculate Total Price per transaction
df_clean = df_clean.copy()
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

# Build RFM table
rfm = df_clean.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalPrice',  'sum')
).reset_index()

print(rfm.shape)
print(rfm.head())

(4338, 4)
   CustomerID  Recency  Frequency  Monetary
0       12346      326          1  77183.60
1       12347        2          7   4310.00
2       12348       75          4   1797.24
3       12349       19          1   1757.55
4       12350      310          1    334.40


In [12]:
print(rfm[['Recency', 'Frequency', 'Monetary']].describe())

           Recency    Frequency       Monetary
count  4338.000000  4338.000000    4338.000000
mean     92.536422     4.272015    2054.266460
std     100.014169     7.697998    8989.230441
min       1.000000     1.000000       3.750000
25%      18.000000     1.000000     307.415000
50%      51.000000     2.000000     674.485000
75%     142.000000     5.000000    1661.740000
max     374.000000   209.000000  280206.020000


In [13]:
from scipy import stats

# Remove outliers
Q1 = rfm[['Recency', 'Frequency', 'Monetary']].quantile(0.05)
Q3 = rfm[['Recency', 'Frequency', 'Monetary']].quantile(0.95)

rfm_clean = rfm[
    (rfm['Recency']   >= Q1['Recency'])   & (rfm['Recency']   <= Q3['Recency'])   &
    (rfm['Frequency'] >= Q1['Frequency']) & (rfm['Frequency'] <= Q3['Frequency']) &
    (rfm['Monetary']  >= Q1['Monetary'])  & (rfm['Monetary']  <= Q3['Monetary'])
]

print("Before outlier removal:", rfm.shape)
print("After outlier removal:", rfm_clean.shape)
print("\nUpdated Stats:")
print(rfm_clean[['Recency', 'Frequency', 'Monetary']].describe())

Before outlier removal: (4338, 4)
After outlier removal: (3535, 4)

Updated Stats:
           Recency    Frequency     Monetary
count  3535.000000  3535.000000  3535.000000
mean     84.097313     3.153324  1077.425724
std      82.956051     2.541834  1070.391619
min       3.000000     1.000000   112.320000
25%      21.000000     1.000000   332.650000
50%      52.000000     2.000000   673.100000
75%     126.000000     4.000000  1427.090000
max     312.000000    13.000000  5756.890000


In [14]:
scaler = StandardScaler()

rfm_scaled = scaler.fit_transform(rfm_clean[['Recency', 'Frequency', 'Monetary']])

# Convert to dataframe for easy viewing
rfm_scaled_df = pd.DataFrame(rfm_scaled, columns=['Recency', 'Frequency', 'Monetary'])

print("Scaled Data Sample:")
print(rfm_scaled_df.head())

Scaled Data Sample:
    Recency  Frequency  Monetary
0 -0.109680   0.333144  0.672573
1 -0.784831  -0.847273  0.635488
2  2.723546  -0.847273 -0.694261
3 -0.579875   1.907033  1.334854
4  1.783156  -0.847273  0.001845


In [16]:
inertia = []
k_range = range(2, 11)

for k in k_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(rfm_scaled)
    inertia.append(kmeans.inertia_)

# Plot Elbow Curve
fig = px.line(
    x=list(k_range),
    y=inertia,
    markers=True,
    title='Elbow Method — Optimal Number of Clusters',
    labels={'x': 'Number of Clusters (K)', 'y': 'Inertia'}
)
fig.update_traces(line_color='royalblue', marker=dict(size=8, color='red'))
fig.write_html("elbow_curve.html")
print("Elbow curve saved! Open elbow_curve.html to view it.")
print("Inertia values:", inertia)

Elbow curve saved! Open elbow_curve.html to view it.
Inertia values: [5878.941203629725, 3444.283372897897, 2640.3059681775235, 2278.34828162146, 1975.8838199651277, 1691.05780451092, 1544.809055375721, 1422.2981141693717, 1324.9475653451768]


In [17]:
silhouette_scores = []

for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(rfm_scaled)
    score = silhouette_score(rfm_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k} → Silhouette Score: {score:.4f}")

K=2 → Silhouette Score: 0.4647
K=3 → Silhouette Score: 0.4695
K=4 → Silhouette Score: 0.4277
K=5 → Silhouette Score: 0.4098
K=6 → Silhouette Score: 0.3761
K=7 → Silhouette Score: 0.3736
K=8 → Silhouette Score: 0.3394
K=9 → Silhouette Score: 0.3462
K=10 → Silhouette Score: 0.3127


In [18]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm_clean = rfm_clean.copy()
rfm_clean['Cluster'] = kmeans.fit_predict(rfm_scaled)

print("Cluster distribution:")
print(rfm_clean['Cluster'].value_counts().sort_index())

Cluster distribution:
Cluster
0     346
1    1622
2     787
3     780
Name: count, dtype: int64


In [19]:
cluster_summary = rfm_clean.groupby('Cluster')[['Recency', 'Frequency', 'Monetary']].mean().round(2)
print(cluster_summary)

         Recency  Frequency  Monetary
Cluster                              
0          29.94       8.55   3470.87
1          51.14       1.92    550.92
2          40.17       4.86   1699.54
3         220.96       1.61    482.89


In [20]:
cluster_labels = {
    0: 'Champions',
    1: 'Occasional Buyers',
    2: 'Loyal Customers',
    3: 'Lost Customers'
}

rfm_clean['Segment'] = rfm_clean['Cluster'].map(cluster_labels)

print(rfm_clean['Segment'].value_counts())
print(rfm_clean.head())

Segment
Occasional Buyers    1622
Loyal Customers       787
Lost Customers        780
Champions             346
Name: count, dtype: int64
   CustomerID  Recency  Frequency  Monetary  Cluster            Segment
2       12348       75          4   1797.24        2    Loyal Customers
3       12349       19          1   1757.55        1  Occasional Buyers
4       12350      310          1    334.40        3     Lost Customers
5       12352       36          8   2506.04        0          Champions
7       12354      232          1   1079.40        3     Lost Customers


In [21]:
import joblib

BASE_DIR = r"C:\Users\satya\Documents\customer-segmentation"
model_path = os.path.join(BASE_DIR, 'models', 'kmeans_model.pkl')
scaler_path = os.path.join(BASE_DIR, 'models', 'scaler.pkl')

joblib.dump(kmeans, model_path)
joblib.dump(scaler, scaler_path)

print("✅ KMeans model saved!")
print("✅ Scaler saved!")

✅ KMeans model saved!
✅ Scaler saved!


In [22]:
pca = PCA(n_components=3)
rfm_pca = pca.fit_transform(rfm_scaled)

print("Explained Variance Ratio:", pca.explained_variance_ratio_)
print("Total Variance Explained:", sum(pca.explained_variance_ratio_).round(4))

Explained Variance Ratio: [0.65887555 0.25304141 0.08808305]
Total Variance Explained: 1.0


In [24]:
pca_df = pd.DataFrame(rfm_pca, columns=['PCA1', 'PCA2', 'PCA3'])
pca_df['Segment'] = rfm_clean['Segment'].values

fig = px.scatter_3d(
    pca_df,
    x='PCA1', y='PCA2', z='PCA3',
    color='Segment',
    title='Customer Segments — 3D PCA Visualization',
    color_discrete_map={
        'Champions':        '#2ecc71',
        'Loyal Customers':  '#3498db',
        'Occasional Buyers':'#f39c12',
        'Lost Customers':   '#e74c3c'
    },
    opacity=0.7
)
fig.update_traces(marker=dict(size=3))
fig.write_html("customer_segments_3d.html")
print("✅ 3D plot saved!")

✅ 3D plot saved!


In [25]:
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=5)
rfm_clean = rfm_clean.copy()
rfm_clean['DBSCAN_Cluster'] = dbscan.fit_predict(rfm_scaled)

print("DBSCAN Cluster Distribution:")
print(rfm_clean['DBSCAN_Cluster'].value_counts().sort_index())

DBSCAN Cluster Distribution:
DBSCAN_Cluster
-1      37
 0    3498
Name: count, dtype: int64
